# One Frame Is Enough: fine-tune YuNet on Colab

**First time:** Runtime > Change runtime type > **GPU**. Upload `yunet_ft.zip` (made on the Mac from `data/yunet_ft`) to `MyDrive/yunet_ft.zip`. Run all.

Starts from the released `weights/yunet_n.pth` (fine-tune, not resume), trains on WIDER FACE train + ChokePoint TRAIN (ChokePoint repeated so it is about half of every epoch), 40 epochs, lr = 1/10 of the trainer default, checkpoint every 5 epochs into `MyDrive/yunet_ft_work` (work dir on Drive), then exports every checkpoint to ONNX and checks the output names.

**After a disconnect:** Run all again. The data is rebuilt on the local disk (~10 min); the training cell resumes from `latest.pth` on Drive.

**Trainer facts (read from the code, `libfacedetection.train`):**
- The CLI has no pretrained-weights flag. `--resume` loads `state_dict` and also takes `epoch` (start epoch = epoch + 1) from the file. So the notebook first writes a weights-only file (`init.pth`: `state_dict` only, no `epoch`, no optimizer) and passes it to `--resume`: epoch counter and optimizer start fresh, weights are the released ones.
- One label file + one image root: `--ann-file` and `--img-prefix` (labelv2: `# path w h` header, then `x1 y1 x2 y2` + 5 keypoints x,y,flag per face; `path` is relative to the root). There is no second-dataset flag, so ChokePoint is merged into the WIDER train labelv2 file.
- Train size: `--image-size N` is a single int, **square**. Training draws a random square crop and resizes it to N x N. 480x640 cannot be set; train at 640 (the net is fully convolutional, the export is dynamic, and the board runs 480x640).
- `--lr` (default 0.01), `--epochs`, `--lr-steps` (epochs), `--checkpoint-interval` (writes `epoch_<N>.pth` plus `latest.pth` every epoch), `--eval-interval 0` turns validation-loss off.
- ONNX export names, in order: `cls_8 cls_16 cls_32 obj_8 obj_16 obj_32 bbox_8 bbox_16 bbox_32 kps_8 kps_16 kps_32`; input `input`.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/yunet_ft_work'          # checkpoints, logs, ONNX: survives disconnects
os.makedirs(WORK, exist_ok=True)
assert os.path.exists(f'{DRIVE}/yunet_ft.zip'), 'upload yunet_ft.zip to MyDrive first'

In [ ]:
# Step 1: trainer
%cd /content
!git clone -q https://github.com/ShiqiYu/libfacedetection.train.git
%cd /content/libfacedetection.train
!pip install -q -e ".[dev]"
!python -m yunet_train.cli.train --help | head -60

In [ ]:
# WIDER FACE train/val images (HuggingFace mirror of the official zips, ~1.8 GB) -> data/widerface/WIDER_{train,val}/images
%cd /content/libfacedetection.train
!mkdir -p data/widerface /content/dl
!wget -q -c -O /content/dl/WIDER_train.zip https://huggingface.co/datasets/CUHK-CSE/wider_face/resolve/main/data/WIDER_train.zip
!wget -q -c -O /content/dl/WIDER_val.zip https://huggingface.co/datasets/CUHK-CSE/wider_face/resolve/main/data/WIDER_val.zip
!unzip -q -o /content/dl/WIDER_train.zip -d data/widerface
!unzip -q -o /content/dl/WIDER_val.zip -d data/widerface
!ls data/widerface

In [ ]:
# labelv2 labels (SCRFD / InsightFace Google Drive file). If gdown fails (Drive quota), download it by hand in a browser
# (https://github.com/deepinsight/insightface/tree/master/detection/scrfd, "annotation files") and save it as MyDrive/wider_labelv2.zip
import glob, shutil
os.chdir('/content/libfacedetection.train')
Z = f'{DRIVE}/wider_labelv2.zip'
if not os.path.exists(Z):
    !pip install -q gdown
    !gdown 1UW3KoApOhusyqSHX96yEDRYiNkd3Iv3Z -O /content/dl/wider_labelv2.zip
    Z = '/content/dl/wider_labelv2.zip'
!rm -rf /content/dl/labelv2 && mkdir -p /content/dl/labelv2
!unzip -q -o "$Z" -d /content/dl/labelv2
os.makedirs('data/widerface/labelv2/train', exist_ok=True); os.makedirs('data/widerface/labelv2/val', exist_ok=True)
for split in ('train', 'val'):
    f = [p for p in glob.glob(f'/content/dl/labelv2/**/{split}/labelv2.txt', recursive=True)]
    assert f, f'no {split}/labelv2.txt inside the zip: look at /content/dl/labelv2 and fix this path'
    shutil.copy(f[0], f'data/widerface/labelv2/{split}/labelv2.txt')
!python -m yunet_train.tools.check_widerface --split train --check-images 10
!python -m yunet_train.tools.check_widerface --split val --check-images 10

In [ ]:
# ChokePoint train/val from the Mac (yunet_ft.zip: train/{images,labelv2.txt}, val/{images,labelv2.txt})
!rm -rf /content/yunet_ft && unzip -q -o "$DRIVE/yunet_ft.zip" -d /content
!ls /content/yunet_ft /content/yunet_ft/train | head
assert os.path.exists('/content/yunet_ft/train/labelv2.txt'), 'zip layout differs: expected /content/yunet_ft/train/labelv2.txt'

## Merge labels: WIDER train + ChokePoint TRAIN (repeated) into one label file and one image root

In [ ]:
import re, random
W = '/content/libfacedetection.train/data/widerface'
CP = '/content/yunet_ft/train'
ROOT = '/content/ft_root'                                  # the single --img-prefix

def blocks(path):
    # one record = '# name w h' header + its face lines
    out, cur = [], None
    for line in open(path):
        if line.startswith('#'):
            cur = [line]; out.append(cur)
        elif line.strip() and cur is not None:
            cur.append(line)
    return out

wider, choke = blocks(f'{W}/labelv2/train/labelv2.txt'), blocks(f'{CP}/labelv2.txt')
reps = max(1, round(len(wider) / len(choke)))              # ChokePoint ~ half of each epoch
print(f'WIDER {len(wider)} records, ChokePoint {len(choke)} x {reps} -> ChokePoint share {len(choke)*reps/(len(wider)+len(choke)*reps):.0%}')

# image root with symlinks: WIDER event folders (0--Parade, ...) and ChokePoint sequence folders (P1E_S1_C1, ...) never share a name
!rm -rf {ROOT} && mkdir -p {ROOT}
for src in (f'{W}/WIDER_train/images', f'{CP}/images'):
    for d in os.listdir(src):
        assert not os.path.exists(f'{ROOT}/{d}'), d
        os.symlink(f'{src}/{d}', f'{ROOT}/{d}')

def write(path, recs):
    with open(path, 'w') as f:
        for r in recs: f.writelines(r)

write('/content/ft_train.txt', wider + choke * reps)
rng = random.Random(0)
write('/content/ft_smoke.txt', rng.sample(wider, 100) + rng.sample(choke, 100))   # 200 images, both sources

# every referenced image must resolve under ROOT
for r in rng.sample(wider, 50) + rng.sample(choke, 50):
    assert os.path.exists(f"{ROOT}/{r[0][1:].split()[0]}"), r[0]
print('merged OK')

## Step 2: smoke run (1 epoch, 200 images)
Expect: `loss=` lines every 2 steps that fall within the epoch, `saved_checkpoint`. Warm-up is shortened to 5 iterations so the lr is not ~0 for a 13-step epoch.

In [ ]:
%cd /content/libfacedetection.train
import torch
os.makedirs(f'{WORK}/smoke', exist_ok=True)
# weights-only init: drops the 'epoch' key so --resume does not skip epochs (the CLI has no pretrained-weights flag)
sd = torch.load('weights/yunet_n.pth', map_location='cpu', weights_only=False)
sd = sd.get('state_dict', sd)
torch.save({'state_dict': sd}, f'{WORK}/init.pth')
!python -m yunet_train.cli.train --variant yunet_n --resume {WORK}/init.pth --ann-file /content/ft_smoke.txt --img-prefix {ROOT} \
    --image-size 640 --epochs 1 --batch-size 16 --workers 2 --device cuda --lr 0.001 --warmup-iters 5 --log-interval 2 \
    --checkpoint-interval 1 --eval-interval 0 --no-tensorboard --work-dir {WORK}/smoke
assert os.path.exists(f'{WORK}/smoke/epoch_1.pth')

## Step 3: fine-tune (40 epochs, 640x640 crops, lr 0.001, checkpoint every 5)
lr = 1/10 of the trainer default 0.01; `--lr-steps 30 36` (epochs) decays it x0.1 twice (the default 400/544 would never trigger in 40 epochs).
shortcut: no degradation augmentation in this run; add blur/dark/downscale augmentation when the stress tier (docs/plan.md Phase 6) exists.
Safe to re-run after a disconnect: it resumes from `latest.pth` (which keeps optimizer, lr schedule and epoch).

In [ ]:
%cd /content/libfacedetection.train
FT = f'{WORK}/yunet_ft'
start = f'{FT}/latest.pth' if os.path.exists(f'{FT}/latest.pth') else f'{WORK}/init.pth'
print('starting from', start)
!python -m yunet_train.cli.train --variant yunet_n --resume {start} --ann-file /content/ft_train.txt --img-prefix {ROOT} \
    --image-size 640 --epochs 40 --batch-size 16 --workers 2 --device cuda --lr 0.001 --lr-steps 30 36 \
    --checkpoint-interval 5 --eval-interval 0 --no-tensorboard --log-interval 100 --work-dir {FT}

## Step 4: export every checkpoint to ONNX (dynamic H/W) and check the output names

In [ ]:
%cd /content/libfacedetection.train
import glob
ckpts = sorted(glob.glob(f'{FT}/epoch_*.pth'), key=lambda p: int(re.findall(r'epoch_(\d+)', p)[0]))
print([os.path.basename(c) for c in ckpts])
for c in ckpts:
    n = re.findall(r'epoch_(\d+)', c)[0]
    !python -m yunet_train.cli.export_onnx {c} --variant yunet_n --dynamic-export --verify --output-file {WORK}/yunet_ft_e{n}.onnx

In [ ]:
import onnx
EXPECT = [f'{h}_{s}' for h in ('cls', 'obj', 'bbox', 'kps') for s in (8, 16, 32)]   # 2023mar order
for f in sorted(glob.glob(f'{WORK}/yunet_ft_e*.onnx')):
    m = onnx.load(f)
    names = [o.name for o in m.graph.output]
    assert names == EXPECT, (f, names)
    assert m.graph.input[0].name == 'input', f
print(len(glob.glob(f'{WORK}/yunet_ft_e*.onnx')), 'ONNX files, outputs', EXPECT)

In [ ]:
# also a copy of the last weights for the repo; download the yunet_ft_e*.onnx files and yunet_ft.pth from MyDrive/yunet_ft_work
shutil.copy(f'{FT}/latest.pth', f'{WORK}/yunet_ft.pth')
!ls -la {WORK}

## Step 6: select the checkpoint and threshold on val (on the Mac, never on test)

Download `yunet_ft_e5.onnx ... yunet_ft_e40.onnx` (and `yunet_ft.pth`) from `MyDrive/yunet_ft_work` into `models/` of the repo. Then, from the repo root:

```bash
# FP32 ORT; the benchmark reads models/yunet_ft.onnx, so copy each checkpoint into that name.
# models/yunet_ft_dyn.onnx is cached by file stem: delete it between checkpoints or the previous one is reused.
for N in 5 10 15 20 25 30 35 40; do
  cp models/yunet_ft_e$N.onnx models/yunet_ft.onnx; rm -f models/yunet_ft_dyn.onnx
  for T in 0.5 0.6 0.7 0.8 0.9; do
    .venv/bin/python src/benchmark.py --split val --systems yunet_ft --thr yunet_ft=$T --out results/ft_val/e$N
  done
done
```

Pick the checkpoint and T with the lowest % identities exposed; ties go to the higher T (less area masked). Then:

```bash
cp models/yunet_ft_e<N>.onnx models/yunet_ft.onnx; rm -f models/yunet_ft_dyn.onnx
# set T in src/systems/detectors.py: "yunet_ft": (..., <T>)  and delete the `shortcut:` placeholder comment
```

Record the val table below (checkpoint x T -> % identities exposed, % area masked) once it exists.